# Vehicle Detection with YOLOv8m (Kaggle)

**Dataset:** [pkdarabi/vehicle-detection-image-dataset](https://www.kaggle.com/datasets/pkdarabi/vehicle-detection-image-dataset)
**5 Classes:** Bus, Car, Motorcycle, Pickup, Truck
**Experiment:** YOLOv8m — Color images

Pipeline:
- Custom augmentation (Albumentations)
- Class-balanced loss weighting
- Multiscale training
- Custom post-processing (Soft-NMS, class-aware thresholds, JSON export)

## 1. Install & Import

In [ ]:
!pip install ultralytics albumentations seaborn scikit-image -q

import os, json, yaml, time, random, math, warnings, shutil
import cv2
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from pathlib import Path
from collections import defaultdict
from ultralytics import YOLO

warnings.filterwarnings('ignore')
sns.set_style('darkgrid')
plt.rcParams['figure.dpi'] = 120

print('Libraries loaded successfully!')

## 2. Configuration

In [ ]:
# Paths
BASE_DIR   = Path('/kaggle/input/datasets/pkdarabi/vehicle-detection-image-dataset')
DATA_DIR   = BASE_DIR / 'No_Apply_Grayscale/No_Apply_Grayscale/Vehicles_Detection.v8i.yolov8'

RESULTS_DIR = Path('/kaggle/working/results')
MODELS_DIR  = RESULTS_DIR / 'models'
PLOTS_DIR   = RESULTS_DIR / 'plots'
PRED_DIR    = RESULTS_DIR / 'predictions'
CONFIG_DIR  = RESULTS_DIR / 'configs'

for d in [MODELS_DIR, PLOTS_DIR, PRED_DIR, CONFIG_DIR]:
    os.makedirs(d, exist_ok=True)

# Constants
CLASS_NAMES = ['Bus', 'Car', 'Motorcycle', 'Pickup', 'Truck']
NUM_CLASSES = 5
IMG_SIZE    = 640

# Hyperparameters
EPOCHS     = 50
BATCH_SIZE = 16
DEVICE     = 0
PATIENCE   = 15
WORKERS    = 8
SEED       = 42
MULTISCALE = True

print(f'Dataset  : {DATA_DIR}')
print(f'Results  : {RESULTS_DIR}')
print(f'Device   : {"CPU" if DEVICE == "cpu" else f"GPU {DEVICE}"}')
print(f'Epochs={EPOCHS}  Batch={BATCH_SIZE}  Multiscale={MULTISCALE}')

In [ ]:
# Verify dataset split sizes
for split in ['train', 'valid', 'test']:
    imgs = len(list((DATA_DIR / split / 'images').glob('*.jpg')))
    lbls = len(list((DATA_DIR / split / 'labels').glob('*.txt')))
    print(f'  {split:>5}: {imgs:>3} images, {lbls:>3} labels')

## 3. Utility Functions

In [ ]:
def create_abs_data_yaml(src_yaml, variant_name):
    with open(str(src_yaml)) as f:
        data = yaml.safe_load(f)
    base = Path(src_yaml).parent
    data['train'] = str(base / 'train' / 'images')
    data['val']   = str(base / 'valid' / 'images')
    data['test']  = str(base / 'test'  / 'images')
    dst = CONFIG_DIR / f'{variant_name}.yaml'
    with open(str(dst), 'w') as f:
        yaml.dump(data, f)
    return str(dst)


def save_plot(fig, filename):
    path = PLOTS_DIR / filename
    fig.savefig(str(path), dpi=150, bbox_inches='tight')
    print(f'  [saved] {path}')


print('Utility functions ready.')

## 4. Custom Augmentation (Albumentations)

In [ ]:
import albumentations as A

random.seed(SEED)
np.random.seed(SEED)


def yolo_to_pascal_voc(boxes, img_w=640, img_h=640):
    out = []
    for xc, yc, w, h in boxes:
        out.append([
            (xc - w / 2) * img_w,
            (yc - h / 2) * img_h,
            (xc + w / 2) * img_w,
            (yc + h / 2) * img_h,
        ])
    return out


def pascal_voc_to_yolo(boxes, img_w=640, img_h=640):
    out = []
    for x1, y1, x2, y2 in boxes:
        out.append([
            (x1 + x2) / 2 / img_w,
            (y1 + y2) / 2 / img_h,
            (x2 - x1) / img_w,
            (y2 - y1) / img_h,
        ])
    return out


train_augmentation = A.Compose([
    A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.5, 1.0), ratio=(0.75, 1.33), p=1.0),
    A.MedianBlur(blur_limit=(3, 7), p=1.0),
    A.RandomBrightnessContrast(brightness_limit=0, contrast_limit=0.3, p=1.0),
    A.HorizontalFlip(p=1.0),
], bbox_params=A.BboxParams(format='pascal_voc', min_visibility=0.3, label_fields=['class_labels']))

print(f'Augmentation pipeline: {len(train_augmentation)} transforms')

In [ ]:
train_img_dir = DATA_DIR / 'train' / 'images'
train_lbl_dir = DATA_DIR / 'train' / 'labels'
COLORS = [(255, 0, 0), (0, 255, 0), (0, 0, 255), (255, 255, 0), (255, 0, 255)]

sample_imgs = random.sample(
    sorted(train_img_dir.glob('*.jpg')),
    min(4, len(list(train_img_dir.glob('*.jpg'))))
)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Original (top) vs Augmented (bottom)', fontsize=16, fontweight='bold')


def draw_boxes_pascal(ax, im, boxes, labels):
    ax.imshow(im)
    for box, cls_id in zip(boxes, labels):
        x1, y1, x2, y2 = box[:4]
        color = tuple(c / 255 for c in COLORS[int(cls_id)][::-1])
        ax.add_patch(patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor=color, facecolor='none'
        ))
        ax.text(x1, y1 - 5, CLASS_NAMES[int(cls_id)],
                color=color, fontsize=10, fontweight='bold')
    ax.axis('off')


for i, img_path in enumerate(sample_imgs):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]

    bboxes_yolo, class_labels = [], []
    lbl_path = train_lbl_dir / (img_path.stem + '.txt')
    if lbl_path.exists():
        for line in lbl_path.read_text().splitlines():
            parts = line.split()
            if len(parts) == 5:
                bboxes_yolo.append(list(map(float, parts[1:])))
                class_labels.append(int(parts[0]))

    bboxes_pascal = yolo_to_pascal_voc(bboxes_yolo, W, H)

    draw_boxes_pascal(axes[0, i], img, bboxes_pascal, class_labels)
    axes[0, i].set_title(f'Original: {img_path.name}', fontsize=9)

    aug = train_augmentation(image=img, bboxes=bboxes_pascal, class_labels=class_labels)
    draw_boxes_pascal(axes[1, i], aug['image'], aug['bboxes'], aug['class_labels'])
    axes[1, i].set_title(f'Augmented ({len(aug["bboxes"])} boxes)', fontsize=9)

plt.tight_layout()
plt.show()

## 5. Training Configuration

**Experiment:** YOLOv8m — Color

### Class-Balanced Loss Weighting
Dataset imbalance: Car=73%, Motorcycle=14%, Pickup=10%, Truck=2%, Bus=1%

In [ ]:
# Class counts from EDA
class_counts = np.array([42, 2600, 485, 339, 75])  # Bus, Car, Motorcycle, Pickup, Truck
class_freq   = class_counts / class_counts.sum()
class_weights = 1.0 / (class_freq + 1e-6)
class_weights /= class_weights.min()
CLASS_LOSS_WEIGHTS = class_weights.tolist()

print('Class weights (custom loss):')
for name, count, w in zip(CLASS_NAMES, class_counts, CLASS_LOSS_WEIGHTS):
    print(f'  {name:>12s}: {count:>5d} samples  weight = {w:.3f}')
print(f'
Mean weight: {np.mean(CLASS_LOSS_WEIGHTS):.3f}')

In [ ]:
custom_loss_log = defaultdict(list)


def class_balanced_loss_callback(trainer):
    if not hasattr(trainer, 'loss_items') or trainer.loss_items is None:
        return
    if isinstance(trainer.loss_items, torch.Tensor):
        loss  = trainer.loss_items.detach().cpu()
        avg_w = np.mean(CLASS_LOSS_WEIGHTS)
        custom_loss_log['default_loss'].append(loss.sum().item())
        custom_loss_log['weighted_loss'].append(
            loss[0].item() + loss[1].item() * avg_w + loss[2].item()
        )


yolo_aug_params = {
    'hsv_h': 0.015, 'hsv_s': 0.7, 'hsv_v': 0.4,
    'degrees': 15.0, 'scale': 0.5, 'translate': 0.1,
    'shear': 5.0, 'flipud': 0.0, 'fliplr': 0.5,
    'mosaic': 1.0, 'mixup': 0.1, 'copy_paste': 0.1,
    'erasing': 0.4, 'auto_augment': 'randaugment',
}

EXP_NAME  = 'yolov8m_color'
EXP_MODEL = 'yolov8m.pt'

print(f'Callback ready  | Multiscale: {MULTISCALE}')
print(f'Experiment      : {EXP_NAME} ({EXP_MODEL})')

## 6. Training

In [ ]:
print('=' * 60)
print(f'Training: {EXP_NAME}')
print('=' * 60)

abs_yaml = create_abs_data_yaml(DATA_DIR / 'data.yaml', EXP_NAME)

model = YOLO(EXP_MODEL)
model.add_callback('on_train_batch_end', class_balanced_loss_callback)

start = time.time()
results = model.train(
    data=abs_yaml,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=WORKERS,
    project=str(MODELS_DIR),
    name=EXP_NAME,
    exist_ok=True,
    patience=PATIENCE,
    seed=SEED,
    verbose=True,
    multi_scale=MULTISCALE,
    **yolo_aug_params,
)
elapsed = time.time() - start
print(f'Completed in {elapsed / 60:.2f} min')

trained_info = {
    'save_dir': str(MODELS_DIR / EXP_NAME),
    'best_pt' : str(MODELS_DIR / EXP_NAME / 'weights' / 'best.pt'),
    'last_pt' : str(MODELS_DIR / EXP_NAME / 'weights' / 'last.pt'),
    'training_time_min': round(elapsed / 60, 2),
}

## 7. Validation & Test Evaluation

In [ ]:
CONF_THRESH = 0.25
IOU_THRESH  = 0.45

print('=' * 60)
print(f'Evaluating: {EXP_NAME}')
print('=' * 60)

if not os.path.exists(trained_info['best_pt']):
    print(f'[!] Weights not found: {trained_info["best_pt"]}')
else:
    model    = YOLO(trained_info['best_pt'])
    abs_yaml = create_abs_data_yaml(DATA_DIR / 'data.yaml', f'{EXP_NAME}_eval')

    print('
[VALIDATION]')
    val_res = model.val(
        data=abs_yaml, split='val', imgsz=IMG_SIZE,
        batch=BATCH_SIZE, device=DEVICE,
        conf=CONF_THRESH, iou=IOU_THRESH, verbose=False,
    )
    print(f'  mAP50={val_res.box.map50:.4f}  mAP50-95={val_res.box.map:.4f}')
    print(f'  Precision={val_res.box.mp:.4f}  Recall={val_res.box.mr:.4f}')

    print('
[TEST]')
    test_res = model.val(
        data=abs_yaml, split='test', imgsz=IMG_SIZE,
        batch=BATCH_SIZE, device=DEVICE,
        conf=CONF_THRESH, iou=IOU_THRESH, verbose=False,
    )
    print(f'  mAP50={test_res.box.map50:.4f}  mAP50-95={test_res.box.map:.4f}')
    print(f'  Precision={test_res.box.mp:.4f}  Recall={test_res.box.mr:.4f}')

eval_results = {
    'val': {
        'mAP50'    : round(float(val_res.box.map50), 4),
        'mAP50-95' : round(float(val_res.box.map),   4),
        'precision': round(float(val_res.box.mp),     4),
        'recall'   : round(float(val_res.box.mr),     4),
    },
    'test': {
        'mAP50'    : round(float(test_res.box.map50), 4),
        'mAP50-95' : round(float(test_res.box.map),   4),
        'precision': round(float(test_res.box.mp),     4),
        'recall'   : round(float(test_res.box.mr),     4),
    },
    'training_time_min': trained_info['training_time_min'],
}

## 8. Custom Post-Processing

In [ ]:
# Per-class confidence thresholds — lower for minority classes
CLASS_CONF_THRESH = [0.15, 0.30, 0.25, 0.20, 0.15]  # Bus, Car, Motorcycle, Pickup, Truck


def soft_nms(boxes, scores, sigma=0.5, score_thresh=0.001):
    if len(boxes) == 0:
        return [], []
    boxes, scores = np.array(boxes), np.array(scores)
    x1, y1, x2, y2 = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]
    areas = (x2 - x1) * (y2 - y1)
    order = scores.argsort()[::-1]
    keep_boxes, keep_scores = [], []
    while order.size > 0:
        i = order[0]
        keep_boxes.append(boxes[i])
        keep_scores.append(scores[i])
        xx1 = np.maximum(x1[i], x1[order[1:]])
        yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]])
        yy2 = np.minimum(y2[i], y2[order[1:]])
        w = np.maximum(0.0, xx2 - xx1)
        h = np.maximum(0.0, yy2 - yy1)
        ious = (w * h) / (areas[i] + areas[order[1:]] - (w * h) + 1e-8)
        scores[order[1:]] *= np.exp(-(ious ** 2) / sigma)
        order = order[np.where(scores[order[1:]] > score_thresh)[0] + 1]
    return keep_boxes, keep_scores


def custom_postprocess(results, class_names, conf_threshs,
                        min_area=400, min_aspect=0.3, max_aspect=4.0):
    all_detections = []
    for result in results:
        if result.boxes is None or len(result.boxes) == 0:
            all_detections.append({'image': result.path, 'detections': []})
            continue
        boxes   = result.boxes.xyxy.cpu().numpy()
        confs   = result.boxes.conf.cpu().numpy()
        cls_ids = result.boxes.cls.cpu().numpy().astype(int)
        detections = []
        for cls_id in np.unique(cls_ids):
            mask      = cls_ids == cls_id
            thresh    = conf_threshs[cls_id]
            cls_boxes = boxes[mask][confs[mask] >= thresh]
            cls_confs = confs[mask][confs[mask] >= thresh]
            if len(cls_boxes) == 0:
                continue
            kept_boxes, kept_scores = soft_nms(cls_boxes, cls_confs)
            for box, score in zip(kept_boxes, kept_scores):
                x1, y1, x2, y2 = box
                bw, bh = x2 - x1, y2 - y1
                area   = bw * bh
                aspect = bw / bh if bh > 0 else 0
                if area < min_area or not (min_aspect <= aspect <= max_aspect):
                    continue
                detections.append({
                    'class_id'    : int(cls_id),
                    'class_name'  : class_names[cls_id],
                    'confidence'  : float(score),
                    'bbox'        : [int(x1), int(y1), int(x2), int(y2)],
                    'area_px'     : int(area),
                    'aspect_ratio': round(float(aspect), 3),
                })
        all_detections.append({
            'image'         : str(result.path),
            'filename'      : Path(result.path).name,
            'num_detections': len(detections),
            'detections'    : detections,
        })
    return all_detections


print('Post-processing functions ready.')

In [ ]:
print('Running custom post-processing on test set...')

if os.path.exists(trained_info['best_pt']):
    model        = YOLO(trained_info['best_pt'])
    test_img_dir = DATA_DIR / 'test' / 'images'

    raw       = model.predict(source=str(test_img_dir), conf=0.001, iou=0.9,
                               device=DEVICE, save=False, verbose=False)
    processed = custom_postprocess(raw, CLASS_NAMES, CLASS_CONF_THRESH)

    export_path = PRED_DIR / f'{EXP_NAME}_predictions.json'
    with open(export_path, 'w') as f:
        json.dump(processed, f, indent=2)

    total_det = sum(p['num_detections'] for p in processed)
    print(f'  {len(processed)} images | {total_det} detections  saved to {export_path.name}')

## 9. Results Summary

In [ ]:
summary = {
    'experiment'  : EXP_NAME,
    'epochs'      : EPOCHS,
    'batch_size'  : BATCH_SIZE,
    'imgsz'       : IMG_SIZE,
    'multi_scale' : MULTISCALE,
    'custom_loss' : {
        'type'   : 'class_balanced_weighting',
        'weights': CLASS_LOSS_WEIGHTS,
    },
    'post_processing': {
        'soft_nms_sigma'       : 0.5,
        'class_conf_thresholds': CLASS_CONF_THRESH,
        'min_area'             : 400,
        'aspect_range'         : [0.3, 4.0],
    },
    'results': eval_results,
}

summary_path = RESULTS_DIR / 'training_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

v, t = eval_results['val'], eval_results['test']
print(f"{'Model':<20} {'Val mAP50':>10} {'Val mAP95':>10} {'Test mAP50':>10} {'Test mAP95':>10} {'Time(min)':>10}")
print('-' * 70)
print(f"{EXP_NAME:<20} {v['mAP50']:>10.4f} {v['mAP50-95']:>10.4f} {t['mAP50']:>10.4f} {t['mAP50-95']:>10.4f} {eval_results['training_time_min']:>10.2f}")
print(f'
Summary saved: {summary_path}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
metrics = ['mAP50', 'mAP50-95', 'precision', 'recall']
vals    = [eval_results['test'][m] for m in metrics]
colors  = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']

bars = ax.bar(metrics, vals, color=colors, edgecolor='black', width=0.5)
for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_title(f'{EXP_NAME} — Test Set Results', fontsize=14, fontweight='bold')
ax.set_ylim(0, max(vals) * 1.25 if vals else 1)
plt.tight_layout()
save_plot(fig, 'results.png')
plt.show()

## 10. Prediction Visualisation

In [ ]:
if os.path.exists(trained_info['best_pt']):
    model        = YOLO(trained_info['best_pt'])
    test_img_dir = DATA_DIR / 'test' / 'images'
    test_images  = sorted(test_img_dir.glob('*.jpg'))[:6]

    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle(f'{EXP_NAME} — Test Set Predictions', fontsize=16, fontweight='bold')

    for ax, img_path in zip(axes.flat, test_images):
        result = model.predict(
            source=str(img_path), conf=CONF_THRESH, iou=IOU_THRESH,
            device=DEVICE, verbose=False, save=False,
        )[0]
        ax.imshow(cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB))
        ax.set_title(img_path.name, fontsize=9)
        ax.axis('off')

    plt.tight_layout()
    save_plot(fig, 'test_predictions.png')
    plt.show()

## 11. Export Outputs (Kaggle)

In [ ]:
print('=' * 60)
print('TRAINING COMPLETE')
print('=' * 60)
print(f'Results dir  : {RESULTS_DIR}')
print(f'Model weights: {MODELS_DIR}')
print(f'Summary      : {summary_path}')
print(f'Predictions  : {PRED_DIR}')

In [ ]:
import zipfile

zip_path = '/kaggle/working/all_outputs.zip'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk('/kaggle/working'):
        for fname in files:
            if fname.endswith('.ipynb') or fname == 'all_outputs.zip':
                continue
            fp = Path(root) / fname
            zf.write(fp, fp.relative_to('/kaggle/working'))

print(f'ZIP ready: {zip_path}')
print('Right-click -> Download, or use the Kaggle API.')